# 10 — Pipeline completo

**Trilha:** [07 contrato](07_contrato_e_contexto.ipynb) → [08 políticas](08_capacidades_e_politicas.ipynb) → [09 runtime](09_runtime_com_dataclasses.ipynb) → **10 pipeline completo**

**De onde viemos:** cada notebook resolveu um pedaço — contrato com contexto (07), permissões (08), custo e histórico (09). O
[09](09_runtime_com_dataclasses.ipynb) terminou mostrando que configuração vinda de fora precisa de validação. Aqui as peças
viram **um fluxo**: configuração do ambiente, entrada validada, classificação, rota, lista com contexto, cache, log sem dados
sensíveis — e testes.

As peças de 07–09 estão em [`mercado.py`](mercado.py), sem mudança de lógica.

In [1]:
import hashlib
import json
import os
import re
from contextlib import contextmanager
from dataclasses import dataclass, field, replace
from decimal import Decimal
from typing import assert_never

from dotenv import find_dotenv
from pydantic import AliasChoices, BaseModel, Field, SecretStr, ValidationError
from pydantic_settings import BaseSettings, SettingsConfigDict

from contrato import ListaDeCompras, Unidade, extrair_json, prompt_com_schema
from llm_simulado import ClienteSimulado
from mercado import (Classificacao, ContextoPedido, Decisao, Ferramenta, IntencaoAberta, ListaComContexto, Perfil, Uso,
                     decidir, INTENCOES_VISTAS_DESCONHECIDAS)

## 1. Configuração vinda do ambiente

`BaseSettings` (pacote `pydantic-settings`) lê variáveis de ambiente e o `.env`, e **valida** como qualquer modelo Pydantic —
o problema da pegadinha 5 do 09 some. Os nomes do seu `.env` (`openai`, `openai_base_url`) funcionam via `AliasChoices`, ao
lado dos nomes com prefixo (`COMPRAS_API_KEY`). A chave é `SecretStr`: não aparece em `repr`, `print` nem JSON.

In [2]:
class Configuracoes(BaseSettings):
    model_config = SettingsConfigDict(
        env_prefix="COMPRAS_",
        env_file=find_dotenv(usecwd=True) or None,  # procura o .env subindo a partir da pasta atual
        extra="ignore",
        validate_by_name=True,  # aceita Configuracoes(api_key=...) — ver pegadinha 3
    )
    api_key: SecretStr = Field(validation_alias=AliasChoices("COMPRAS_API_KEY", "openai"))
    base_url: str = Field(default="https://api.openai.com/v1",
                          validation_alias=AliasChoices("COMPRAS_BASE_URL", "openai_base_url"))
    modelo: str = "gpt-4o-mini"
    max_tentativas: int = Field(default=3, ge=1, le=5)
    orcamento_padrao: Decimal = Field(default=Decimal("50"), gt=0)


@contextmanager
def variaveis_de_ambiente(**valores: str):
    """Define variáveis só dentro do bloco — para demonstrar sem mexer no seu ambiente."""
    antigos = {nome: os.environ.get(nome) for nome in valores}
    os.environ.update(valores)
    try:
        yield
    finally:
        for nome, antigo in antigos.items():
            if antigo is None:
                os.environ.pop(nome, None)
            else:
                os.environ[nome] = antigo


with variaveis_de_ambiente(openai="sk-exemplo-123", openai_base_url="https://meu-endpoint/v1", COMPRAS_MAX_TENTATIVAS="2"):
    cfg = Configuracoes()
print(f"cfg: {cfg!r}")
print(f"cfg.model_dump_json(): {cfg.model_dump_json()}")

with variaveis_de_ambiente(openai="sk-exemplo-123", COMPRAS_MAX_TENTATIVAS="9"):
    try:
        Configuracoes()
    except ValidationError as ex:
        print(f"COMPRAS_MAX_TENTATIVAS=9: {ex.errors()[0]['msg']}")

cfg: Configuracoes(api_key=SecretStr('**********'), base_url='https://meu-endpoint/v1', modelo='gpt-4o-mini', max_tentativas=2, orcamento_padrao=Decimal('50'))
cfg.model_dump_json(): {"api_key":"**********","base_url":"https://meu-endpoint/v1","modelo":"gpt-4o-mini","max_tentativas":2,"orcamento_padrao":"50"}
COMPRAS_MAX_TENTATIVAS=9: Input should be less than or equal to 5


## 2. Entrada validada

O que o usuário manda também é fronteira: um modelo Pydantic para o pedido. O perfil vem da **sessão** (quem está logado), não do
texto.

In [3]:
class Pedido(BaseModel):
    texto: str = Field(min_length=1, max_length=500)
    perfil: Perfil
    porcoes: int = Field(default=2, ge=1, le=20)
    orcamento: Decimal | None = Field(default=None, gt=0)


try:
    Pedido(texto="", perfil="administrador", porcoes=50)
except ValidationError as ex:
    for erro in ex.errors():
        print(f"{erro['loc'][0]}: {erro['msg']}")

texto: String should have at least 1 character
perfil: Input should be 'visitante' or 'cliente'
porcoes: Input should be less than or equal to 20


## 3. Log sem dados sensíveis

Tudo que vai para log passa por `redigir`: e-mails e sequências que parecem número de cartão viram máscara. É uma rede de
segurança simples — não substitui não logar o que não precisa (pegadinha 5).

In [4]:
PADROES_SENSIVEIS = [
    (re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+"), "<email>"),
    (re.compile(r"\b(?:\d[ -]?){13,19}\b"), "<cartao>"),
]


def redigir(texto: str) -> str:
    for padrao, mascara in PADROES_SENSIVEIS:
        texto = padrao.sub(mascara, texto)
    return texto


print(f'redigir("Lista de pizza, manda pra ana@exemplo.com e paga no 4111 1111 1111 1111"): {redigir("Lista de pizza, manda pra ana@exemplo.com e paga no 4111 1111 1111 1111")}')

redigir("Lista de pizza, manda pra ana@exemplo.com e paga no 4111 1111 1111 1111"): Lista de pizza, manda pra <email> e paga no <cartao>


## 4. O pipeline

Etapas: chave de idempotência → cache → classificar → rotear (com a política do 08) → gerar lista com contexto (07) → registrar
uso (09). O `match` em `IntencaoAberta` cobre todos os membros, incluindo `DESCONHECIDO`, e termina em `assert_never`.

In [5]:
INSTRUCAO_CLASSIFICAR = ("Classifique o pedido de um assistente de compras. "
                         "Preencha 'prato' quando houver um prato e 'ingrediente' quando houver um ingrediente.")
INSTRUCAO_LISTA = "Você gera listas de compras com ingredientes in natura e sistema métrico, em quantidades realistas."
LIMIAR_CONFIANCA = 0.7


@dataclass(frozen=True)
class Resultado:
    intencao: IntencaoAberta
    resposta: str
    custo_lista: Decimal | None = None
    do_cache: bool = False


@dataclass
class Pipeline:
    cliente: object
    cfg: Configuracoes
    precos: dict
    despensa: dict = field(default_factory=dict)
    cache: dict[str, Resultado] = field(default_factory=dict)
    uso: Uso = field(default_factory=Uso)
    log: list[str] = field(default_factory=list)

    def chave(self, pedido: Pedido) -> str:
        normalizado = pedido.model_dump(mode="json") | {"texto": " ".join(pedido.texto.lower().split())}
        return hashlib.sha256(json.dumps(normalizado, sort_keys=True).encode()).hexdigest()[:16]

    def _chamar(self, instrucao: str, texto: str) -> str:
        resposta = self.cliente.chat.completions.create(
            model=self.cfg.modelo,
            messages=[{"role": "system", "content": instrucao}, {"role": "user", "content": texto}],
        )
        self.uso.registrar(resposta.usage)
        return resposta.choices[0].message.content

    def _registrar(self, evento: str) -> None:
        self.log.append(redigir(evento))

    def executar(self, pedido: Pedido) -> Resultado:
        chave = self.chave(pedido)
        self._registrar(f"pedido {chave} [{pedido.perfil}]: {pedido.texto}")
        if chave in self.cache:
            self._registrar(f"pedido {chave}: cache")
            return replace(self.cache[chave], do_cache=True)
        resultado = self._processar(pedido)
        self.cache[chave] = resultado
        self._registrar(f"pedido {chave}: {resultado.intencao} -> {resultado.resposta[:60]}")
        return resultado

    def _processar(self, pedido: Pedido) -> Resultado:
        texto = self._chamar(prompt_com_schema(INSTRUCAO_CLASSIFICAR, Classificacao), pedido.texto)
        c = Classificacao.model_validate_json(extrair_json(texto))
        if c.confianca < LIMIAR_CONFIANCA:
            return Resultado(c.intencao, f"não tenho certeza do pedido (confiança {c.confianca:.2f}); pode reformular?")
        match c.intencao:
            case IntencaoAberta.RECEITA:
                return self._receita(pedido, c)
            case IntencaoAberta.PRECO:
                return self._preco(pedido, c)
            case IntencaoAberta.FORA_DO_ESCOPO:
                return Resultado(c.intencao, "só ajudo com compras de mercado")
            case IntencaoAberta.DESCONHECIDO:
                return Resultado(c.intencao, "não entendi o pedido; pode reformular?")
            case _:
                assert_never(c.intencao)

    def _preco(self, pedido: Pedido, c: Classificacao) -> Resultado:
        decisao, motivo = decidir(pedido.perfil, Ferramenta.CONSULTAR_PRECO)
        if decisao is not Decisao.PERMITIR:
            return Resultado(c.intencao, f"não posso consultar preço: {motivo}")
        if c.ingrediente not in self.precos:
            return Resultado(c.intencao, f"não tenho preço de {c.ingrediente!r}")
        preco, unidade = self.precos[c.ingrediente]
        return Resultado(c.intencao, f"{c.ingrediente}: R$ {preco} / {unidade}")

    def _receita(self, pedido: Pedido, c: Classificacao) -> Resultado:
        ctx = ContextoPedido(orcamento=pedido.orcamento or self.cfg.orcamento_padrao, precos=self.precos,
                             despensa=self.despensa)
        instrucao = prompt_com_schema(INSTRUCAO_LISTA, ListaDeCompras) + f"\nO custo não pode passar de R$ {ctx.orcamento}."
        erros = []
        for _ in range(self.cfg.max_tentativas):
            pedido_texto = f"{c.prato} para {pedido.porcoes} pessoas" + (f"\nCorrija: {erros[-1]}" if erros else "")
            try:
                lista = ListaComContexto.model_validate_json(extrair_json(self._chamar(instrucao, pedido_texto)),
                                                             context=ctx)
            except ValidationError as ex:
                erros.append(ex.errors()[0]["msg"])
                continue
            return Resultado(c.intencao, f"lista de {lista.prato}: {len(lista.ingredientes)} itens", lista.custo)
        return Resultado(c.intencao, f"não consegui montar a lista: {erros[-1]}")

## 5. Rodando com o modelo simulado

O modelo simulado responde conforme o prompt de sistema: classifica por palavra-chave e, quando pedem lista, devolve uma lista
fixa (*respostas construídas*).

In [6]:
PRECOS = {
    "farinha de trigo": (Decimal("6.50"), Unidade.KG),
    "muçarela de búfala": (Decimal("89.90"), Unidade.KG),
    "tomate italiano": (Decimal("1.20"), Unidade.UN),
    "azeite": (Decimal("45.00"), Unidade.L),
}
DESPENSA = {"azeite": (Decimal("0.5"), Unidade.L)}

LISTA_PIZZA = json.dumps({"prato": "pizza margherita", "porcoes": 2, "ingredientes": [
    {"nome": "farinha de trigo", "quantidade": 250, "unidade": "g", "categoria": "mercearia"},
    {"nome": "muçarela de búfala", "quantidade": 200, "unidade": "g", "categoria": "laticinios"},
    {"nome": "tomate italiano", "quantidade": 4, "unidade": "un", "categoria": "hortifruti"},
    {"nome": "azeite", "quantidade": 30, "unidade": "ml", "categoria": "mercearia"},
]}, ensure_ascii=False)

CLASSIFICACOES = {
    "pizza": '{"intencao": "receita", "confianca": 0.95, "prato": "pizza margherita"}',
    "quanto": '{"intencao": "preco", "confianca": 0.9, "ingrediente": "azeite"}',
    "janta": '{"intencao": "receita", "confianca": 0.4}',
    "filme": '{"intencao": "fora_do_escopo", "confianca": 0.97}',
}


def modelo_simulado(mensagens):
    sistema, usuario = mensagens[0]["content"], mensagens[-1]["content"].lower()
    if sistema.startswith(INSTRUCAO_LISTA):
        return LISTA_PIZZA
    for palavra, resposta in CLASSIFICACOES.items():
        if palavra in usuario:
            return resposta
    return '{"intencao": "cardapio_semanal", "confianca": 0.8}'


cfg_demo = Configuracoes(api_key="sk-simulado")
cliente = ClienteSimulado(padrao=modelo_simulado)
pipeline = Pipeline(cliente, cfg_demo, PRECOS, DESPENSA)

pedidos = [
    Pedido(texto="Pizza margherita, por favor", perfil="cliente"),
    Pedido(texto="Quanto está o azeite?", perfil="visitante"),
    Pedido(texto="Tem algo pra janta?", perfil="cliente"),
    Pedido(texto="Me indica um filme", perfil="cliente"),
    Pedido(texto="Monta o cardápio da semana", perfil="cliente"),
    Pedido(texto="  PIZZA margherita,   por favor ", perfil="cliente"),
]
for pedido in pedidos:
    r = pipeline.executar(pedido)
    custo = f" | custo R$ {r.custo_lista}" if r.custo_lista is not None else ""
    print(f"{pedido.texto.strip()[:28]:<30} [{r.intencao}{', cache' if r.do_cache else ''}] {r.resposta}{custo}")

Pizza margherita, por favor    [receita] lista de pizza margherita: 4 itens | custo R$ 24.41
Quanto está o azeite?          [preco] azeite: R$ 45.00 / l
Tem algo pra janta?            [receita] não tenho certeza do pedido (confiança 0.40); pode reformular?
Me indica um filme             [fora_do_escopo] só ajudo com compras de mercado
Monta o cardápio da semana     [desconhecido] não entendi o pedido; pode reformular?
PIZZA margherita,   por favo   [receita, cache] lista de pizza margherita: 4 itens | custo R$ 24.41


In [7]:
print(f"chamadas ao modelo: {pipeline.uso.chamadas}")
print("  = 2 na receita (classificar + gerar lista) + 1 em cada um dos outros 4 + 0 no último (cache)")
print(f"tokens (estimados): entrada {pipeline.uso.tokens_entrada}, saída {pipeline.uso.tokens_saida}")
print(f"intenções desconhecidas registradas: {INTENCOES_VISTAS_DESCONHECIDAS}")

chamadas ao modelo: 6
  = 2 na receita (classificar + gerar lista) + 1 em cada um dos outros 4 + 0 no último (cache)
tokens (estimados): entrada 1334, saída 173
intenções desconhecidas registradas: ['cardapio_semanal']


## 6. Testes

Com o cliente simulado, o pipeline vira código testável como qualquer outro. Cada teste é uma função com `assert` — o mesmo
formato que o `pytest` executa.

In [8]:
def novo_pipeline(**kwargs) -> Pipeline:
    return Pipeline(ClienteSimulado(padrao=modelo_simulado), cfg_demo, PRECOS, DESPENSA, **kwargs)


def teste_idempotencia_nao_chama_o_modelo_de_novo():
    p = novo_pipeline()
    p.executar(Pedido(texto="Pizza", perfil="cliente"))
    chamadas = p.uso.chamadas
    assert p.executar(Pedido(texto="pizza ", perfil="cliente")).do_cache
    assert p.uso.chamadas == chamadas


def teste_orcamento_apertado_recusa_a_lista():
    r = novo_pipeline().executar(Pedido(texto="Pizza", perfil="cliente", orcamento=Decimal("10")))
    assert r.custo_lista is None and "orçamento" in r.resposta, r.resposta


def teste_matriz_de_politica():
    esperado = {
        (Perfil.VISITANTE, Ferramenta.FINALIZAR_PEDIDO): Decisao.NEGAR,
        (Perfil.VISITANTE, Ferramenta.CONSULTAR_PRECO): Decisao.PERMITIR,
        (Perfil.CLIENTE, Ferramenta.FINALIZAR_PEDIDO): Decisao.CONFIRMAR,
    }
    for (perfil, ferramenta), decisao in esperado.items():
        assert decidir(perfil, ferramenta)[0] is decisao, (perfil, ferramenta)


def teste_log_nao_vaza_email():
    p = novo_pipeline()
    p.executar(Pedido(texto="Pizza e manda pra ana@exemplo.com", perfil="cliente"))
    assert not any("ana@exemplo.com" in linha for linha in p.log)


def teste_chave_nao_aparece_em_repr():
    assert "sk-simulado" not in repr(cfg_demo) and "sk-simulado" not in cfg_demo.model_dump_json()


for teste in [teste_idempotencia_nao_chama_o_modelo_de_novo, teste_orcamento_apertado_recusa_a_lista,
              teste_matriz_de_politica, teste_log_nao_vaza_email, teste_chave_nao_aparece_em_repr]:
    teste()
    print(f"ok  {teste.__name__}")

ok  teste_idempotencia_nao_chama_o_modelo_de_novo
ok  teste_orcamento_apertado_recusa_a_lista
ok  teste_matriz_de_politica
ok  teste_log_nao_vaza_email
ok  teste_chave_nao_aparece_em_repr


## 7. Trocando pelo modelo real

O `Pipeline` só usa `cliente.chat.completions.create` — a mesma interface do `OpenAI(...)`. Trocar é uma linha (célula de API:
não roda sem `.env`):

In [ ]:
# api: o mesmo pipeline com o cliente real
from openai import OpenAI

cfg = Configuracoes()  # lê openai / openai_base_url do seu .env
cliente_real = OpenAI(api_key=cfg.api_key.get_secret_value(), base_url=cfg.base_url)
pipeline_real = Pipeline(cliente_real, cfg, PRECOS, DESPENSA)
for texto in ["Pizza margherita para 2", "Quanto está o azeite?", "Me indica um filme"]:
    r = pipeline_real.executar(Pedido(texto=texto, perfil="cliente"))
    print(f"{texto} -> [{r.intencao}] {r.resposta}")
print(f"uso: {pipeline_real.uso}")

## ⚠️ O que ainda quebra

**1. A chave de cache precisa incluir tudo que muda a resposta.** A `chave` usa o pedido inteiro (texto, perfil, porções,
orçamento). Se usasse só o texto, um visitante receberia do cache uma resposta montada para um cliente — e uma pergunta de
orçamento diferente receberia a lista antiga:

In [9]:
def chave_so_texto(pedido: Pedido) -> str:
    return " ".join(pedido.texto.lower().split())


a = Pedido(texto="Pizza", perfil="cliente", orcamento=Decimal("100"))
b = Pedido(texto="pizza", perfil="visitante", orcamento=Decimal("10"))
print(f"chave só com texto — iguais? {chave_so_texto(a) == chave_so_texto(b)}")
print(f"chave do pipeline  — iguais? {pipeline.chave(a) == pipeline.chave(b)}")

chave só com texto — iguais? True
chave do pipeline  — iguais? False


E cache não tem validade aqui: se o preço da muçarela mudar, o custo em cache fica velho. Em produção, cada entrada precisa de
prazo de validade (TTL) ou de invalidação quando os preços mudam.

**2. `str(SecretStr)` é a máscara, não a chave.** Passar `str(cfg.api_key)` para o cliente envia `**********` — e o erro que
volta é de autenticação, longe da causa. Use `get_secret_value()`, só no ponto em que a chave é necessária:

In [10]:
print(f"str(cfg_demo.api_key): {str(cfg_demo.api_key)}")
print(f"f'{{cfg_demo.api_key}}': {cfg_demo.api_key}")
print(f"cfg_demo.api_key.get_secret_value(): {cfg_demo.api_key.get_secret_value()}")

str(cfg_demo.api_key): **********
f'{cfg_demo.api_key}': **********
cfg_demo.api_key.get_secret_value(): sk-simulado


**3. `validation_alias` desliga o nome do campo — e `extra="ignore"` esconde isso.** Sem o `validate_by_name=True`,
`Configuracoes(api_key="...")` **ignora o argumento em silêncio** e usa o valor do ambiente. Testado:

In [11]:
class ConfiguracoesSemNome(BaseSettings):
    model_config = SettingsConfigDict(env_prefix="COMPRAS_", extra="ignore")
    api_key: SecretStr = Field(validation_alias=AliasChoices("COMPRAS_API_KEY", "openai"))


with variaveis_de_ambiente(openai="sk-do-ambiente"):
    sem_nome = ConfiguracoesSemNome(api_key="sk-passada-no-codigo")
    com_nome = Configuracoes(api_key="sk-passada-no-codigo")
print(f"sem validate_by_name: {sem_nome.api_key.get_secret_value()}")
print(f"com validate_by_name: {com_nome.api_key.get_secret_value()}")

sem validate_by_name: sk-do-ambiente
com validate_by_name: sk-passada-no-codigo


**4. Testes com modelo simulado testam o encanamento, não o modelo.** Os testes da seção 6 provam que cache, política, orçamento
e log funcionam. Não dizem se o modelo real classifica bem ou gera listas boas. Para isso, o mesmo formato de teste roda contra o
cliente real, com um conjunto de pedidos e respostas esperadas — e mede **taxa de acerto**, não "passou/falhou" (o modelo varia
entre execuções).

**5. `redigir` é rede de segurança, não garantia.** Expressão regular deixa passar formatos que ela não conhece:

In [12]:
for texto in ["cartão 4111-1111-1111-1111", "cartão 4111.1111.1111.1111", "email ana arroba exemplo ponto com"]:
    print(f"{texto!r} -> {redigir(texto)!r}")

'cartão 4111-1111-1111-1111' -> 'cartão <cartao>'
'cartão 4111.1111.1111.1111' -> 'cartão 4111.1111.1111.1111'
'email ana arroba exemplo ponto com' -> 'email ana arroba exemplo ponto com'


A defesa principal é **não registrar** o texto bruto quando não precisa (registrar a chave do pedido e a intenção basta para
a maioria das análises).

## Fim da trilha

| Nível | Notebooks | Ideia central |
|---|---|---|
| Básico | 01 → 03 | chamar a API, contrato com Pydantic, vocabulário com enum, guardrail de escopo |
| Intermediário | 04 → 06 | ferramentas como funções, regras + retry, roteamento por enum |
| Avançado | 07 → 10 | contrato com contexto, permissões, runtime com dataclasses, pipeline testável |

A divisão de trabalho que atravessou a trilha: **Pydantic na fronteira** (o que entra e sai do modelo, configuração, pedido do
usuário), **dataclass no estado interno** (configuração congelada, histórico, uso, resultados), **enum para todo vocabulário
fechado** (unidades, intenções, ferramentas, perfis, decisões) — e **código, não prompt,** para tudo que precisa ser garantido.

A versão anterior destes exemplos (mais densa, sem a progressão) está em [`casos_originais/`](../casos_originais/README.md).